# **Задача классификации твитов о стихийных бедствиях**

In [ ]:
import torch

print(torch.cuda.is_available())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords


In [ ]:
df_test = pd.read_csv('/content/test.csv')
df_train = pd.read_csv('/content/train.csv')

In [ ]:
df_train.shape

In [ ]:
df_train.info()

В "Keyword" и "location" имеются пропуски ,которые необходимо обработать

In [ ]:
df_train.head(10)

In [ ]:
df_train['text'].duplicated().sum()

In [ ]:
target_conflicts = df_train.groupby('text')['target'].nunique()

In [ ]:
target_conflicts[target_conflicts > 1]

В датасете есть 110 повторяющихся текстов . Большинство из них имеют одинаковую разметку, но около 10 текстов встречаются с разными значениями **target**, что указывает на противаоречивую разметку

In [ ]:
df_train.drop_duplicates(subset='text',inplace=True)

In [ ]:
df_train['text'].duplicated().sum()

Из-за малого количества конфликтующие строки удалены, обычные дубликаты сведены к одному экземпляру.

In [ ]:
df_train.isna().sum()

In [ ]:
df_train['target'].value_counts()

In [ ]:
plt.pie(df_train['target'].value_counts(),autopct='%1.1f%%');

Присутствует небольшой дисбаланс классовс 57/43


Посчитаем длину каждого твита в символах

In [ ]:
df_train['text_len_chars'] = df_train['text'].str.len()

In [ ]:
df_train['text_len_chars']

In [ ]:
df_train['text_len_chars'].describe()

Большинство текстов короткие,что ожидаемо для твитов. Медиана равна 107, а максимальная 157.Сильных выбросов визуально не ожидается , но то стоит проверить распределением.

Посчитаем колличество слов в каждом твите

In [ ]:
df_train['text_len_words'] = df_train['text'].str.split().map(lambda x: len(x))

In [ ]:
df_train['text_len_words'].describe()

In [ ]:
df_train.groupby('target')['text_len_words'].agg(['mean', 'median'])

Посмотрим частоту слов во всем документе

In [ ]:
df_train['text'].str.lower().str.split().explode().value_counts()

При сыром анализе , большую часть занимают стоп-слова, для качественной проверки частоты слов , следует убрать их

In [ ]:
nltk.download('stopwords')
stop = set(stopwords.words('english'))

In [ ]:
words = df_train['text'].str.lower().str.split().explode()

words[~words.isin(stop)].value_counts()

в текстах есть URL ,HTML- сущности,пунктуационный шум

In [ ]:
df_test.info()

In [ ]:
df_test.shape

In [ ]:
df_test['text'].duplicated().sum()

In [ ]:
df_test[df_test['text'].duplicated(keep=False)].sort_values('text')

**Выводы по EDA:**
- В обучающем датасете около 7.5 тыс. объектов после очистки дубликатов.
- Целевая переменная распределена примерно как 57% для класса 0 и 43% для класса 1, поэтому сильного дисбаланса нет.
- Пропуски есть в keyword и особенно в location, но в text и target пропусков нет. Для первого baseline можно использовать только text, поэтому эти пропуски пока не критичны.
- Были обнаружены повторяющиеся тексты, в том числе небольшое число конфликтующих примеров с разными target; они были удалены из train.
- Тексты короткие: медианная длина около 15 слов и 107 символов. Средняя длина у классов почти не отличается, поэтому сама длина текста вряд ли является сильным признаком.
- В частотном анализе сырых текстов доминируют стоп-слова, также встречаются URL, mentions, HTML-сущности и пунктуационный шум. Это нужно учитывать в preprocessing-экспериментах.
- В test также есть повторяющиеся тексты, но их не удаляем, чтобы сохранить исходную структуру и соответствие id для финального submission.

Следующий этап: разделение train на train/validation и построение первого baseline TF-IDF + Logistic Regression.

## **Baseline**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df_train['text']
y = df_train['target']
RANDOM_STATE = 42

X_train,X_valid,y_train,y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tf_idf = TfidfVectorizer()

X_train_tfidf = tf_idf.fit_transform(X_train)
X_valid_tfidf = tf_idf.transform(X_valid)

In [ ]:
feature_names = tf_idf.get_feature_names_out()
print(feature_names)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression(random_state=RANDOM_STATE)

lr.fit(X_train_tfidf,y_train)

lr_v0_pred = lr.predict(X_valid_tfidf)

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
f1 = f1_score(y_valid, lr_v0_pred)

print(f1)

**Baseline v0:** raw text + default TF-IDF + Logistic Regression

**F1:** 0.7515

In [ ]:
import re
import string

In [ ]:
def clear_text(text):
  text = re.sub(r'https?://\S+|www\.\S+', '', text)
  text =re.sub(r'@\w+', '', text)

  return text


In [ ]:
df_base = df_train.copy()

df_base['text'] = df_base['text'].apply(clear_text)

In [ ]:
mask = df_train['text'].str.contains(r'https?://|www\.|@\w+',regex=True)

df_train.loc[mask,['text']].head()

In [ ]:
df_base.loc[mask,['text']].head()

In [ ]:
X = df_base['text']
y = df_base['target']
RANDOM_STATE = 42

X_train_v1,X_valid_v1,y_train_v1,y_valid_v1 = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE)

In [ ]:
tf_idf_v1 = TfidfVectorizer()

X_train_tfidf_v1 = tf_idf_v1.fit_transform(X_train_v1)
X_valid_tfidf_v1 = tf_idf_v1.transform(X_valid_v1)

In [ ]:
lr_v1 = LogisticRegression(random_state=RANDOM_STATE)

lr_v1.fit(X_train_tfidf_v1,y_train_v1)

pred_v1 = lr_v1.predict(X_valid_tfidf_v1)

f1_v1 = f1_score(y_valid_v1, pred_v1)

print('F1-score:',f1_v1)

Удаление URL и mentions дало небольшой прирост в качестве модели , но он минемален. Значит эти элементы действительно вноят немного шума , но не являются главным фактором качества


| Версия | Preprocessing | F1 |
|---|---|---:|
| v0 | raw text | 0.7512 |
| v1 | remove URL + mentions | 0.7525 |

Проверим качество после удаления стоп-слов

In [ ]:
def remove_stopwords(text):

  words = text.lower().split()
  filtered_words = [word for word in words if word not in stop]

  return ' '.join(filtered_words)

df_stopwords = df_base.copy()

df_stopwords['text'] = df_stopwords['text'].apply(remove_stopwords)

In [ ]:
X = df_stopwords['text']
y = df_stopwords['target']
RANDOM_STATE = 42

X_train_v2,X_valid_v2,y_train_v2,y_valid_v2 = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE)

In [ ]:
tf_idf_v2 = TfidfVectorizer()

X_train_tfidf_v2 = tf_idf_v2.fit_transform(X_train_v2)
X_valid_tfidf_v2 = tf_idf_v2.transform(X_valid_v2)

In [ ]:
lr_v2 = LogisticRegression(random_state=RANDOM_STATE)

lr_v2.fit(X_train_tfidf_v2,y_train_v2)

pred_v2 = lr_v2.predict(X_valid_tfidf_v2)

f1_v2 = f1_score(y_valid_v2, pred_v2)

print('F1-score:',f1_v2)

В этой задаче stopwords ,похоже, несут полезный сигнал , поэтому их удаление ухудшает F1

| Версия | Preprocessing | F1 |
|---|---|---:|
| v0 | raw text | 0.7512 |
| v1 | remove URL + mentions | 0.7525 |
| v2 | v1 + remove stopwords | 0.7439 |

Используем лемматизацию для дальнейшей оценки качества модели после preprocessing

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm')

def lemmatize_text(text):

  doc = nlp(text)
  lemmas = [token.lemma_ for token in doc]

  return " ".join(lemmas)

Используем библиотеку  **scapy** так как она, в отличии от **WordNetLemmatizer**, сама сопоставляет части речи к словам

In [ ]:
df_lemma = df_base.copy()

df_lemma['text'] = df_lemma['text'].apply(lemmatize_text)

In [ ]:
X = df_lemma['text']
y = df_lemma['target']
RANDOM_STATE = 42

X_train_v3,X_valid_v3,y_train_v3,y_valid_v3 = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE)

In [ ]:
tf_idf_v3 = TfidfVectorizer()

X_train_tfidf_v3 = tf_idf_v3.fit_transform(X_train_v3)
X_valid_tfidf_v3 = tf_idf_v3.transform(X_valid_v3)

In [ ]:
lr_v3 = LogisticRegression(random_state=RANDOM_STATE)

lr_v3.fit(X_train_tfidf_v3,y_train_v3)

pred_v3 = lr_v3.predict(X_valid_tfidf_v3)

f1_v3 = f1_score(y_valid_v3, pred_v3)

print('F1-score:',f1_v3)

Лемматизация дала небольшой прирост, это еще раз подтверждает что стоп-слова для этой задачи несут полезную информацию , взглянем на работу **стемминга** по сравнению с **лемматизацией**.

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

In [ ]:
def stemming_text(text):

  words = text.lower().split()
  stemmer_words = [stemmer.stem(word) for word in words]

  return " ".join(stemmer_words)

In [ ]:
df_stem = df_base.copy()
df_stem['text'] = df_stem['text'].apply(stemming_text)

In [ ]:
X = df_stem['text']
y = df_stem['target']
RANDOM_STATE = 42

X_train_v4,X_valid_v4,y_train_v4,y_valid_v4 = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE)

In [ ]:
tf_idf_v4 = TfidfVectorizer()

X_train_tfidf_v4 = tf_idf_v4.fit_transform(X_train_v4)
X_valid_tfidf_v4 = tf_idf_v4.transform(X_valid_v4)

In [ ]:
lr_v4 = LogisticRegression(random_state=RANDOM_STATE)

lr_v4.fit(X_train_tfidf_v4,y_train_v4)

pred_v4 = lr_v4.predict(X_valid_tfidf_v4)

f1_v4 = f1_score(y_valid_v4, pred_v4)

print('F1-score:',f1_v4)

Лемматизация показала лучший результат среди рассмотренных вариантов предобработки. Stemming также дал небольшой прирост относительно базовой очистки, однако оказался немного хуже лемматизации. Удаление стоп-слов, наоборот, ухудшило F1-score.

Проверим насколько уверена модель в своих предсказаниях, почему такие результаты мы получили


In [ ]:
proba = lr_v3.predict_proba(X_valid_tfidf_v3)[:,1]

In [ ]:
analysis = pd.DataFrame({
    "text": X_valid_v3.values,
    "y_true": y_valid_v3.values,
    "y_pred": pred_v3,
    "scores": proba
})

In [ ]:
analysis

In [ ]:
analysis['dist_to_threshold'] = abs(analysis['scores'] - 0.5)

In [ ]:
tp = analysis[(analysis['y_true'] == 1)&(analysis['y_pred'] == 1)]

In [ ]:
pd.set_option("display.max_colwidth", None)

In [ ]:
tp.sort_values("dist_to_threshold")[
    ["text", "y_true", "y_pred", "scores"]
].head(10)

In [ ]:
def explain_tweet(text):
  x =tf_idf_v3.transform([text])

  feature_names = tf_idf_v3.get_feature_names_out()
  weights = lr_v3.coef_[0]

  indices = x.nonzero()[1]

  rows = []

  for i in indices:
    tfidf_value = x[0,i]
    weight = weights[i]
    contribution = tfidf_value * weight

    rows.append({
        "word": feature_names[i],
        "tfidf": tfidf_value,
        "weight": weight,
        "contribution": contribution
    })

  return pd.DataFrame(rows).sort_values(
      "contribution",
      ascending = False
  )

In [ ]:
text = tp.sort_values("dist_to_threshold").iloc[0]['text']
explain_tweet(text)

In [ ]:
text = tp.sort_values("dist_to_threshold").iloc[1]["text"]
explain_tweet(text)

In [ ]:
text = tp.sort_values("dist_to_threshold").iloc[2]["text"]
explain_tweet(text)

In [ ]:
lr_v3.intercept_[0]

In [ ]:
tn = analysis[(analysis['y_true'] == 0)&(analysis['y_pred'] == 0)]

In [ ]:
tn.sort_values("dist_to_threshold")[
    ["text", "y_true", "y_pred", "scores"]
].head(10)

In [ ]:
text = tn.sort_values('dist_to_threshold').iloc[0]["text"]

explain_tweet(text)

In [ ]:
fn = analysis[(analysis['y_true'] == 1)&(analysis['y_pred'] == 0)]

In [ ]:
fn.sort_values("dist_to_threshold")[
    ["text", "y_true", "y_pred", "scores"]
].head(10)

In [ ]:
text = fn.sort_values('dist_to_threshold').iloc[0]["text"]

explain_tweet(text)

In [ ]:
text = fn.sort_values('dist_to_threshold').iloc[1]["text"]

explain_tweet(text)

Модель правильно отнесла твит к положительному классу, но с низкой уверенностью. Положительные и отрицательные вклады TF-IDF-признаков почти компенсировали друг друга. Суммарный вклад признаков составил около +0.60, однако отрицательный intercept модели около -0.54 уменьшил итоговый логит до примерно 0.06. После применения sigmoid вероятность положительного класса получилась около 0.51, то есть почти на границе threshold 0.5. При этом модель присвоила слову in больший положительный коэффициент, чем слову dead, что показывает ограничение TF-IDF + Logistic Regression: модель опирается на статистические корреляции признаков, а не на полноценное понимание контекста

## **XGBClassifier**

Проверим качество на ансомблевых моделях с кастомными параметрами

In [ ]:
from xgboost import XGBRFClassifier

xgb = XGBRFClassifier(random_state=RANDOM_STATE)

In [ ]:
xgb.fit(X_train_tfidf_v3,y_train_v3)

pred_v3 = xgb.predict(X_valid_tfidf_v3)

f1_v3 = f1_score(y_valid_v3, pred_v3)

print('F1-score:',f1_v3)

Линейная модель значительно превзошла классический градиентный бустинг на TF-IDF-признаках, что связано с высокой размерностью и разреженностью текстового представления.

# **DL + Classic ML**

Загрузим выбранную модель

In [ ]:
!pip install sentence_transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'Tarka-AIR/Tarka-Embedding-150M-V1',
    device="cuda"
    )

embedding = model.encode("A building collapsed after the storm")
print(embedding.shape)

In [ ]:
train_embeddings = model.encode(
    X_train.to_list(),
    batch_size=32,
    show_progress_bar=True
)

In [ ]:
valid_embeddings = model.encode(
    X_valid.to_list(),
    batch_size=32,
    show_progress_bar=True
)


In [ ]:
lr = LogisticRegression(random_state=RANDOM_STATE)

lr.fit(train_embeddings,y_train)

pred = lr.predict(valid_embeddings)
proba = lr.predict_proba(valid_embeddings)[:,1]

f1 = f1_score(y_valid, pred)

print('F1-score:',f1)

В этой модели мы использовали необработанный текст и получили качество **F1 = 0.7871**. Проверим качество на обработанном тексте , используем базовую обработку ( удалим тэги и ссылки)

In [ ]:
train_embeddings_base = model.encode(
    X_train_v1.to_list(),
    batch_size=32,
    show_progress_bar=True
)

valid_embeddings_base = model.encode(
    X_valid_v1.to_list(),
    batch_size=32,
    show_progress_bar=True
)


In [ ]:
lr_base = LogisticRegression(random_state=RANDOM_STATE)

lr_base.fit(train_embeddings_base,y_train_v1)

pred_base = lr_base.predict(valid_embeddings_base)
proba_base = lr.predict_proba(valid_embeddings_base)[:,1]

f1_base = f1_score(y_valid_v1, pred_base)

print('F1-score:',f1_base)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
cm = confusion_matrix(y_valid, pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=lr.classes_)

disp.plot(cmap=plt.cm.Blues)
plt.show()

In [ ]:
cm = confusion_matrix(y_valid_v1, pred_v1)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=lr_v1.classes_)

disp.plot(cmap=plt.cm.Blues)
plt.show()

Использование **transformer embeddings** заметно улучшило распознавание положительного класса. Число ***true positive увеличилось с 444 до 477***, тогда как число ***true negative практически не изменилось — с 765 до 766***. Следовательно, основной прирост F1 связан со снижением количества false negative и улучшением полноты по классу disaster.

# **Fine-tune pre-trained model (трансформер по архитектуре)**

Для fine-tuning была выбрана distilbert-base-uncased. Модель представляет собой компактную версию BERT, сохраняющую encoder-архитектуру и хорошо подходящую для задач классификации текста. По сравнению с BERT она требует меньше вычислительных ресурсов и быстрее обучается, что делает её удобной для экспериментов в Colab. Поскольку датасет содержит короткие англоязычные твиты, использование более крупной модели не было необходимо.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
    )

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=2
    )

In [ ]:
train_encodings = tokenizer(
    X_train.to_list(),
    truncation=True,
    padding=True,
    max_length=128
    )

valid_encodings = tokenizer(
    X_valid.to_list(),
    truncation=True,
    padding=True,
    max_length=128
    )

In [ ]:
train_encodings.keys()

In [ ]:
len(train_encodings['input_ids'])

In [ ]:
X_train.shape

Проверим длину последовательности

In [ ]:
len(train_encodings['input_ids'][0])

In [ ]:
len(train_encodings["attention_mask"][0])

In [ ]:
train_encodings

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': y_train.to_list()
})

valid_dataset = Dataset.from_dict({
    'input_ids': valid_encodings['input_ids'],
    'attention_mask': valid_encodings['attention_mask'],
    'labels': y_valid.to_list()
})

In [ ]:
train_dataset[0]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy='epoch',
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end= True,
    metric_for_best_model="f1",
    greater_is_better=True
)

In [ ]:
import numpy as np

In [ ]:
def compute_metrics(eval_pred):
  predictions,labels = eval_pred
  preds = np.argmax(predictions,axis=1)

  return {
      "f1": f1_score(labels,preds)
  }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset= valid_dataset,
    compute_metrics= compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
pred_output = trainer.predict(valid_dataset)

In [ ]:
pred = pred_output.predictions

In [ ]:
predictions = np.argmax(pred,axis=1)

In [ ]:
cm = confusion_matrix(y_valid,predictions)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=lr_v1.classes_)

disp.plot(cmap=plt.cm.Blues)
plt.show()

## Итоговый вывод

В рамках проекта были последовательно протестированы три подхода к классификации Disaster Tweets.

Первый подход использовал классические NLP-методы: preprocessing, TF-IDF и Logistic Regression. Лучший результат был получен при использовании lemmatization и составил примерно F1 = 0.756. Эксперименты также показали, что удаление stopwords ухудшило качество, а Gradient Boosting значительно уступил линейной модели на высокоразмерных sparse TF-IDF признаках.

На втором этапе TF-IDF был заменён на contextual embeddings модели Tarka-AIR/Tarka-Embedding-150M-V1. Модель была выбрана благодаря хорошим benchmark-результатам на задачах text representation/classification, умеренному размеру около 150M параметров и совместимости с Sentence Transformers. Использование Transformer embeddings позволило повысить F1 примерно до 0.787. При этом дополнительный preprocessing практически не повлиял на результат, поэтому для Transformer-подходов был сохранён исходный текст.

На последнем этапе был выполнен fine-tuning `distilbert-base-uncased`. DistilBERT был выбран как более лёгкая и быстрая версия BERT, сохраняющая большую часть его возможностей при значительно меньшем количестве параметров. Использовался именно base checkpoint, а не модель, ранее fine-tuned на SST-2, чтобы адаптация происходила непосредственно под текущую задачу классификации Disaster Tweets.

Fine-tuning показал лучший результат — около F1 = 0.81. Это подтверждает, что адаптация внутренних representation Transformer под конкретную downstream-задачу позволяет получить дополнительный прирост по сравнению с использованием frozen embeddings.

Итоговая динамика качества:

TF-IDF + Logistic Regression → ~0.756  
Transformer Embeddings + Logistic Regression → ~0.787  
Fine-tuned DistilBERT → ~0.808

Таким образом, каждый переход к более контекстному представлению текста повышал качество классификации. При этом классический TF-IDF остаётся сильным и дешёвым baseline, pretrained embeddings дают хороший компромисс между скоростью и качеством, а fine-tuning Transformer обеспечивает лучший результат ценой большей вычислительной сложности.